# 🌍 Air Quality Data Cleaning & Preprocessing

## Enterprise Data Pipeline - Jupyter Notebook

This notebook handles:
1. **Data Generation** - Generate synthetic monitoring data
2. **Data Cleaning** - Handle missing values, outliers, duplicates
3. **Feature Engineering** - Create derived features
4. **Exploratory Data Analysis (EDA)** - Visualize patterns
5. **Data Export** - Save cleaned datasets for ML models

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded successfully')
print(f'📅 Notebook run: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## 1. 📊 Data Generation
Generate realistic environmental monitoring data using our data generator.

In [ ]:
from backend.utils.data_generator import (
    generate_pollutant_data,
    generate_weather_data,
    generate_sensor_data,
    generate_health_records,
    generate_hospital_data,
    STATIONS
)

# Generate 1 year of data
start_date = datetime.now() - timedelta(days=365)
end_date = datetime.now()

print('Generating pollutant data...')
df_pollutants = generate_pollutant_data(start_date, end_date, freq_hours=1)
print(f'  ✅ Pollutants: {df_pollutants.shape}')

print('Generating weather data...')
df_weather = generate_weather_data(start_date, end_date, freq_hours=1)
print(f'  ✅ Weather: {df_weather.shape}')

print('Generating sensor data...')
df_sensors = generate_sensor_data(num_sensors=20, hours=168)
print(f'  ✅ Sensors: {df_sensors.shape}')

print('Generating health records...')
df_health = generate_health_records(500)
print(f'  ✅ Health: {df_health.shape}')

print('Generating hospital data...')
df_hospitals = generate_hospital_data()
print(f'  ✅ Hospitals: {df_hospitals.shape}')

print('\n🎉 All data generated successfully!')

In [ ]:
# Quick look at pollutant data
print('📋 Pollutant Data Sample:')
print(f'Shape: {df_pollutants.shape}')
print(f'Date Range: {df_pollutants["timestamp"].min()} to {df_pollutants["timestamp"].max()}')
print(f'Stations: {df_pollutants["station_id"].nunique()}')
print(f'Cities: {df_pollutants["city"].unique().tolist()}')
df_pollutants.head()

## 2. 🧹 Data Cleaning
Handle missing values, outliers, duplicates, and data quality issues.

In [ ]:
def clean_pollutant_data(df):
    """Comprehensive data cleaning pipeline for pollutant data."""
    print('🧹 Starting data cleaning pipeline...')
    original_shape = df.shape
    
    # 1. Remove exact duplicates
    df = df.drop_duplicates()
    print(f'  1. Removed {original_shape[0] - len(df)} duplicate rows')
    
    # 2. Check for missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f'  2. Missing values found:')
        print(missing[missing > 0])
    else:
        print(f'  2. No missing values found ✅')
    
    # 3. Handle missing values - forward fill then backward fill for time series
    numeric_cols = ['pm25', 'pm10', 'co', 'so2', 'no2', 'o3', 'aqi']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df.groupby('station_id')[col].transform(
                lambda x: x.fillna(method='ffill').fillna(method='bfill')
            )
    print(f'  3. Forward/backward fill applied for time series continuity')
    
    # 4. Remove physical impossibilities
    for col in numeric_cols:
        if col in df.columns:
            negatives = (df[col] < 0).sum()
            if negatives > 0:
                df.loc[df[col] < 0, col] = 0
                print(f'  4. Fixed {negatives} negative values in {col}')
    print(f'  4. Physical impossibilities removed ✅')
    
    # 5. Detect and cap outliers using IQR method
    outlier_count = 0
    for col in numeric_cols:
        if col in df.columns:
            Q1 = df[col].quantile(0.01)
            Q3 = df[col].quantile(0.99)
            IQR = Q3 - Q1
            lower = max(0, Q1 - 3 * IQR)
            upper = Q3 + 3 * IQR
            outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            outlier_count += outliers
            df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'  5. Capped {outlier_count} outliers using IQR method')
    
    # 6. Ensure timestamps are sorted
    df = df.sort_values(['station_id', 'timestamp']).reset_index(drop=True)
    print(f'  6. Data sorted by station and timestamp')
    
    # 7. Recalculate AQI after cleaning
    from backend.utils.data_generator import calculate_aqi
    df['aqi_recalculated'] = df.apply(
        lambda row: calculate_aqi(row['pm25'], row['pm10'], row['co'], row['so2'], row['no2'], row['o3'])[0],
        axis=1
    )
    df['dominant_pollutant_recalc'] = df.apply(
        lambda row: calculate_aqi(row['pm25'], row['pm10'], row['co'], row['so2'], row['no2'], row['o3'])[1],
        axis=1
    )
    print(f'  7. AQI recalculated from cleaned pollutant values')
    
    print(f'\n✅ Cleaning complete: {original_shape} → {df.shape}')
    return df

df_pollutants_clean = clean_pollutant_data(df_pollutants.copy())

In [ ]:
def clean_weather_data(df):
    """Clean weather data."""
    print('🧹 Cleaning weather data...')
    df = df.drop_duplicates()
    
    # Physical constraints
    df['temperature'] = df['temperature'].clip(-10, 55)  # India range
    df['humidity'] = df['humidity'].clip(0, 100)
    df['pressure'] = df['pressure'].clip(950, 1060)
    df['wind_speed'] = df['wind_speed'].clip(0, 50)
    df['wind_direction'] = df['wind_direction'].clip(0, 360)
    df['precipitation'] = df['precipitation'].clip(0, 200)
    df['uv_index'] = df['uv_index'].clip(0, 15)
    
    # Fill missing values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df[col] = df.groupby('location')[col].transform(
            lambda x: x.fillna(method='ffill').fillna(method='bfill')
        )
    
    df = df.sort_values(['location', 'timestamp']).reset_index(drop=True)
    print(f'✅ Weather data cleaned: {df.shape}')
    return df

df_weather_clean = clean_weather_data(df_weather.copy())

## 3. 🔧 Feature Engineering
Create derived features for ML models.

In [ ]:
def engineer_features(df):
    """Create derived features for ML models."""
    print('🔧 Engineering features...')
    
    # Time-based features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_rush_hour'] = ((df['hour'].between(7, 10)) | (df['hour'].between(17, 21))).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['season'] = df['month'].map({12: 'Winter', 1: 'Winter', 2: 'Winter',
                                     3: 'Spring', 4: 'Spring', 5: 'Summer',
                                     6: 'Monsoon', 7: 'Monsoon', 8: 'Monsoon', 9: 'Monsoon',
                                     10: 'Autumn', 11: 'Autumn'})
    
    # Rolling statistics (per station)
    for col in ['pm25', 'pm10', 'aqi']:
        if col in df.columns:
            df[f'{col}_rolling_24h'] = df.groupby('station_id')[col].transform(
                lambda x: x.rolling(24, min_periods=1).mean()
            )
            df[f'{col}_rolling_7d'] = df.groupby('station_id')[col].transform(
                lambda x: x.rolling(168, min_periods=1).mean()
            )
            df[f'{col}_std_24h'] = df.groupby('station_id')[col].transform(
                lambda x: x.rolling(24, min_periods=1).std()
            )
    
    # Lag features
    for lag in [1, 6, 12, 24]:
        df[f'aqi_lag_{lag}h'] = df.groupby('station_id')['aqi'].shift(lag)
    
    # Rate of change
    df['aqi_change_1h'] = df.groupby('station_id')['aqi'].diff(1)
    df['aqi_change_6h'] = df.groupby('station_id')['aqi'].diff(6)
    df['aqi_change_24h'] = df.groupby('station_id')['aqi'].diff(24)
    
    # Pollution index (normalized composite)
    df['pollution_index'] = (
        df['pm25'] / 250 * 0.3 +
        df['pm10'] / 430 * 0.2 +
        df['co'] / 10 * 0.15 +
        df['no2'] / 180 * 0.15 +
        df['so2'] / 80 * 0.1 +
        df['o3'] / 160 * 0.1
    )
    
    # Simulated traffic density & industrial activity
    np.random.seed(42)
    rush_factor = np.where(df['is_rush_hour'], 0.7, 0.3)
    df['traffic_density'] = np.clip(rush_factor + np.random.normal(0, 0.1, len(df)), 0, 1)
    df['industrial_activity'] = np.clip(0.5 + np.random.normal(0, 0.15, len(df)), 0, 1)
    
    # Fill NaN from lag/rolling with 0
    df = df.fillna(0)
    
    print(f'✅ Feature engineering complete. New shape: {df.shape}')
    print(f'   New features: {df.shape[1] - 14} derived features added')
    return df

df_engineered = engineer_features(df_pollutants_clean.copy())
print(f'\nColumns: {df_engineered.columns.tolist()}')

## 4. 📊 Exploratory Data Analysis (EDA)

In [ ]:
# AQI Distribution by City
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Air Quality Analysis Across Indian Cities', fontsize=16, color='white')

# 1. AQI Distribution
cities = df_engineered['city'].unique()[:8]
for city_name in cities:
    city_data = df_engineered[df_engineered['city'] == city_name]
    axes[0, 0].hist(city_data['aqi'], bins=50, alpha=0.5, label=city_name)
axes[0, 0].set_title('AQI Distribution by City')
axes[0, 0].set_xlabel('AQI')
axes[0, 0].legend(fontsize=8)

# 2. Daily Pattern
hourly_avg = df_engineered.groupby('hour')['aqi'].mean()
axes[0, 1].plot(hourly_avg.index, hourly_avg.values, 'o-', color='#6366f1', linewidth=2)
axes[0, 1].fill_between(hourly_avg.index, hourly_avg.values, alpha=0.3, color='#6366f1')
axes[0, 1].set_title('Average AQI by Hour of Day')
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Average AQI')

# 3. Monthly Pattern
monthly_avg = df_engineered.groupby('month')['aqi'].mean()
axes[1, 0].bar(monthly_avg.index, monthly_avg.values, color='#8b5cf6')
axes[1, 0].set_title('Average AQI by Month')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Average AQI')

# 4. Box plot by city
city_order = df_engineered.groupby('city')['aqi'].median().sort_values(ascending=False).index
data_for_box = [df_engineered[df_engineered['city'] == c]['aqi'].values for c in city_order]
bp = axes[1, 1].boxplot(data_for_box, labels=city_order, patch_artist=True)
colors = ['#6366f1', '#8b5cf6', '#06b6d4', '#10b981', '#f59e0b', '#ef4444', '#ec4899', '#f97316']
for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1, 1].set_title('AQI Distribution by City')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../datasets/eda_aqi_analysis.png', dpi=150, bbox_inches='tight', facecolor='#1a1f3a')
plt.show()
print('📊 EDA plots saved!')

In [ ]:
# Correlation Matrix
numeric_cols = ['pm25', 'pm10', 'co', 'so2', 'no2', 'o3', 'aqi',
                'traffic_density', 'industrial_activity']
corr = df_engineered[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Pollutant Correlation Matrix', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('../datasets/correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor='#1a1f3a')
plt.show()
print('📊 Correlation matrix saved!')

In [ ]:
# Detailed statistics
print('📊 Pollutant Statistics (Cleaned Data):')
print('=' * 80)
stats = df_engineered[['pm25', 'pm10', 'co', 'so2', 'no2', 'o3', 'aqi']].describe().round(2)
print(stats)

print('\n📊 City-wise AQI Summary:')
print('=' * 80)
city_stats = df_engineered.groupby('city')['aqi'].agg(['mean', 'median', 'std', 'min', 'max']).round(1)
city_stats = city_stats.sort_values('mean', ascending=False)
print(city_stats)

## 5. 💾 Data Export
Save cleaned and engineered datasets for ML model training.

In [ ]:
# Create datasets directory
os.makedirs('../datasets', exist_ok=True)

# Save cleaned pollutant data
df_engineered.to_csv('../datasets/sample_aqi_data.csv', index=False)
print(f'✅ Saved: datasets/sample_aqi_data.csv ({df_engineered.shape})')

# Save cleaned weather data
df_weather_clean.to_csv('../datasets/sample_weather_data.csv', index=False)
print(f'✅ Saved: datasets/sample_weather_data.csv ({df_weather_clean.shape})')

# Save health records
df_health.to_csv('../datasets/sample_health_data.csv', index=False)
print(f'✅ Saved: datasets/sample_health_data.csv ({df_health.shape})')

# Save hospital data
df_hospitals.to_csv('../datasets/sample_hospitals.csv', index=False)
print(f'✅ Saved: datasets/sample_hospitals.csv ({df_hospitals.shape})')

# Save station metadata
import json
with open('../datasets/sample_stations.json', 'w') as f:
    json.dump(STATIONS, f, indent=2)
print(f'✅ Saved: datasets/sample_stations.json')

# Save per-station data for LSTM training
os.makedirs('../datasets/per_station', exist_ok=True)
for station_id in df_engineered['station_id'].unique():
    station_data = df_engineered[df_engineered['station_id'] == station_id]
    station_data.to_csv(f'../datasets/per_station/{station_id}.csv', index=False)
print(f'✅ Saved per-station data for {df_engineered["station_id"].nunique()} stations')

print('\n🎉 All datasets exported successfully!')
print(f'Total records: {len(df_engineered) + len(df_weather_clean) + len(df_health):,}')

In [ ]:
# Final Summary
print('=' * 60)
print('📋 DATA PIPELINE SUMMARY')
print('=' * 60)
print(f'📊 Pollutant records: {len(df_engineered):,}')
print(f'🌤️ Weather records:   {len(df_weather_clean):,}')
print(f'🏥 Health records:    {len(df_health):,}')
print(f'🏨 Hospital records:  {len(df_hospitals):,}')
print(f'📡 Stations:          {df_engineered["station_id"].nunique()}')
print(f'🏙️ Cities:            {df_engineered["city"].nunique()}')
print(f'📅 Date Range:        {df_engineered["timestamp"].min()} to {df_engineered["timestamp"].max()}')
print(f'📐 Features:          {df_engineered.shape[1]}')
print(f'\n✅ Data is ready for ML model training!')
print('Next: Run notebooks/02_lstm_training.py or notebooks/03_xgboost_training.py')